# Controlled 5M baseline

This notebook is the audit control. It uses the repaired FEN/action-ID model, not the novel GAVN. Run the smoke test first; only then launch the long job. Every production run uploads complete state to Hugging Face.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time
REPO = Path('/kaggle/working/chess-slm-benchmark')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
SL_REPO = Path('/kaggle/working/searchless_chess')
if not SL_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/google-deepmind/searchless_chess.git', str(SL_REPO)], check=True)
HF_SHARDS = 'chessbench-full-build'  # 8 shards on HF, 5GB peak, no 25GB assemble
assert (REPO / 'scripts/train_student.py').exists(), 'clone failed'
# HF token with retry (Kaggle Secrets service flaps)
for _ in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_WRITE_TOKEN'] = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
        break
    except Exception as exc:
        print(f'HF secret retry in 5s: {exc}')
        time.sleep(5)
if not os.environ.get('HF_WRITE_TOKEN'):
    print('WARNING: HF_WRITE_TOKEN not available after retries, checkpoints will be local only')
os.chdir(REPO)


In [ ]:
# The student trainer runs the official searchless_chess JAX/Haiku stack.
# Isolated era venv: the image ships numpy 2 / pandas built for numpy 2, which
# breaks the era stack's numpy 1.26. GPU comes from the cuda12 pjrt plugin.
import subprocess, sys
V = '/kaggle/working/slvenv'
# the image's python3 lacks ensurepip; use the virtualenv package instead
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'virtualenv'], check=True)
subprocess.run([sys.executable, '-m', 'virtualenv', V], check=True)
subprocess.run([f'{V}/bin/pip', 'install', '--quiet', '-U', 'pip'], check=True)
subprocess.run([f'{V}/bin/pip', 'install', '--quiet',
    '-f', 'https://storage.googleapis.com/jax-releases/jax_releases.html',
    'jax==0.4.35', 'jaxlib==0.4.35', 'jax_cuda12_pjrt==0.4.35', 'jax_cuda12_plugin==0.4.35',
    'orbax-checkpoint==0.5.5', 'dm-haiku==0.0.11', 'numpy==1.26.4', 'pandas==2.2.3',
    'jaxtyping', 'typing-extensions', 'python-chess', 'apache-beam', 'grain',
    'huggingface_hub'], check=True)
print('era venv installed')


In [ ]:
# Smoke test: small data, short run, no HF upload (sharded).
smoke_out = Path('/kaggle/working/baseline-smoke')
cmd = ['/kaggle/working/slvenv/bin/python', 'scripts/train_student.py', '--hf-shards', HF_SHARDS, '--outdir', str(smoke_out), '--sl-repo', str(SL_REPO), '--max-records', '4096', '--steps', '20', '--batch', '64', '--w-rank', '0', '--ckpt-every', '20']
subprocess.run(cmd, check=True)
print('smoke test passed')


In [ ]:
# Change only RUN_ID when launching another independent kernel. Each prefix must be unique.
RUN_ID = 'account1-baseline-5m-seed0'
OUT = Path('/kaggle/working') / RUN_ID
STEPS = 30000
RESUME = True
cmd = ['/kaggle/working/slvenv/bin/python', 'scripts/train_student.py', '--hf-shards', HF_SHARDS, '--outdir', str(OUT), '--sl-repo', str(SL_REPO), '--dim', '224', '--layers', '6', '--heads', '8', '--batch', '512', '--steps', str(STEPS), '--lr', '0.0003', '--warmup', '1000', '--w-kl', '1.0', '--w-ce', '0.5', '--w-rank', '0.5', '--rank-subsample', '2000000', '--ckpt-every', '2000', '--hf-repo', 'vedangfake/chess-slm-benchmark', '--hf-run', RUN_ID, '--hf-upload-every', '1800']
if RESUME:
    cmd.append('--resume-from-hf')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


Do not select a final model from the frozen MATE/puzzle sets here. The output must be evaluated only after training, using the separate evaluation notebook.